# Alternating Least Squares
---
GitHub set-up:

In [1]:
!git clone https://github.com/mkobycheva/recommendation-system.git 2>/dev/null \
  || (cd recommendation-system && git pull)

%cd recommendation-system

import sys
sys.path.insert(0, '.')

from google.colab import drive
drive.mount('/content/drive')

/content/recommendation-system
Mounted at /content/drive


In [2]:
!git checkout als && git pull

Branch 'als' set up to track remote branch 'als' from 'origin'.
Switched to a new branch 'als'
Already up to date.


In [3]:
import pandas as pd

DATA_DIR = '/content/drive/MyDrive/recsys-data'

books_train = pd.read_csv(f'{DATA_DIR}/books_train.csv')
books_valid = pd.read_csv(f'{DATA_DIR}/books_valid.csv')
books_test  = pd.read_csv(f'{DATA_DIR}/books_test.csv')

movies_train = pd.read_csv(f'{DATA_DIR}/movies_train.csv')
movies_valid = pd.read_csv(f'{DATA_DIR}/movies_valid.csv')
movies_test  = pd.read_csv(f'{DATA_DIR}/movies_test.csv')

## Data preparation - matrix construction

In [5]:
%pip install implicit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 57.9 MB/s eta 0:00:00


In [6]:
import implicit
import numpy as np
from scipy.sparse import csr_matrix, hstack

In [9]:
all_users = pd.concat([
    books_train.user_id,
    movies_train.user_id
]).unique()
user2idx = {u: i for i, u in enumerate(all_users)}
n_users  = len(user2idx)

all_users, n_users

(array(['AFSKPY37N3C43SOI5IEXEK5JSIYA', 'AFGNIWCBLQT2QIXXKIW7Q6VREZRQ',
        'AHZ6XMOLEWA67S3TX7IWEXXGWSOA', ...,
        'AERJR6ANVSFEOS2ZXOO4UHGKEXHQ', 'AGRQK3JJ7QUBS3ETIL3VUMTHCNMA',
        'AFTAHGYNIX23U23GYPORLF6GFTBA'], dtype=object),
 127188)

In [10]:
book2idx  = {b: i for i, b in enumerate(books_train.parent_asin.unique())}
movie2idx = {m: i for i, m in enumerate(movies_train.parent_asin.unique())}
n_books  = len(book2idx)
n_movies = len(movie2idx)

n_books, n_movies

(404630, 182434)

In [11]:
books_train['uidx'] = books_train.user_id.map(user2idx)
books_train['iidx'] = books_train.parent_asin.map(book2idx)
movies_train['uidx'] = movies_train.user_id.map(user2idx)
movies_train['iidx'] = movies_train.parent_asin.map(movie2idx)

In [13]:
min(books_train.rating)

1.0